# Spaceship Titanic
## Predict which passengers are transported to an alternate dimension

Bienvenidos al año 2912, donde sus habilidades en ciencia de datos son necesarias para resolver un misterio cósmico. Hemos recibido una transmisión desde cuatro años luz de distancia y la situación no pinta bien.

La nave espacial Titanic era un transatlántico interestelar de pasajeros que fue botado hace un mes. Con casi 13.000 pasajeros a bordo, la nave emprendió su viaje inaugural transportando emigrantes de nuestro sistema solar a tres exoplanetas recientemente habitables que orbitan estrellas cercanas.

Mientras rodeaba Alfa Centauri rumbo a su primer destino —el tórrido 55 Cancri E—, la desprevenida nave espacial Titanic colisionó con una anomalía espaciotemporal oculta en una nube de polvo. Lamentablemente, corrió la misma suerte que su homónima de 1000 años atrás. Aunque la nave permaneció intacta, ¡casi la mitad de los pasajeros fueron transportados a una dimensión alternativa!

Para ayudar a los equipos de rescate y recuperar a los pasajeros desaparecidos, tu reto consiste en predecir qué pasajeros fueron transportados por la anomalía utilizando los registros recuperados del sistema informático dañado de la nave espacial.

¡Ayúdanos a salvarlos y a cambiar la historia!

**PassengerId-** Un ID único para cada pasajero. Cada ID tiene el formato gggg_ppdonde ggggindica el grupo con el que viaja el pasajero y ppes su número dentro del grupo. Las personas en un grupo suelen ser miembros de la misma familia, pero no siempre.

**HomePlanet-** El planeta del que partió el pasajero, normalmente su planeta de residencia permanente.

**CryoSleep-** Indica si el pasajero optó por entrar en animación suspendida durante el viaje. Los pasajeros en criosueño permanecen confinados en sus camarotes.

**Cabin-** El número de camarote donde se aloja el pasajero. Tiene el formato deck/num/side, donde sidepuede ser Ppara babor o Spara estribor.

**Destination-** El planeta al que desembarcará el pasajero.

**Age-** La edad del pasajero.

**VIP-** Si el pasajero pagó por un servicio VIP especial durante el viaje.

**RoomService, FoodCourt, ShoppingMall, Spa, VRDeck-** Cantidad que el pasajero ha facturado en cada una de las numerosas comodidades de lujo del Spaceship Titanic.

**Name-** El nombre y apellido del pasajero.

**Transported-** Si el pasajero fue transportado a otra dimensión. Este es el objetivo, la columna que intentas predecir.

---

### Comienzo importando librerías y cargando el dataset

In [28]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import models, layers

In [13]:
# cargo el archivo CSV en un DataFrame de pandas llamado 'df'
df = pd.read_csv('space_titanic.csv')

In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 1.2+ MB


In [15]:
df.describe()   

,Age,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck
count,8514.000000,8512.000000,8510.000000,8485.000000,8510.000000,8505.000000
mean,28.827930,224.687617,458.077203,173.729169,311.138778,304.854791
std,14.489021,666.717663,1611.489240,604.696458,1136.705535,1145.717189
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,19.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,27.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,38.000000,47.000000,76.000000,27.000000,59.000000,46.000000
max,79.000000,14327.000000,29813.000000,23492.000000,22408.000000,24133.000000


In [16]:
df.head(10)

,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True
5,0005_01,Earth,False,F/0/P,PSO J318.5-22,44.0,False,0.0,483.0,0.0,291.0,0.0,Sandie Hinetthews,True
6,0006_01,Earth,False,F/2/S,TRAPPIST-1e,26.0,False,42.0,1539.0,3.0,0.0,0.0,Billex Jacostaffey,True
7,0006_02,Earth,True,G/0/S,TRAPPIST-1e,28.0,False,0.0,0.0,0.0,0.0,NaN,Candra Jacostaffey,True
8,0007_01,Earth,False,F/3/S,TRAPPIST-1e,35.0,False,0.0,785.0,17.0,216.0,0.0,Andona Beston,True
9,0008_01,Europa,True,B/1/P,55 Cancri e,14.0,False,0.0,0.0,0.0,0.0,0.0,Erraiam Flatic,True


In [17]:
# muestro los nulos de cada variable
print("\nNulos por columna:")
print(df.isnull().sum())


Nulos por columna:
PassengerId       0
HomePlanet      201
CryoSleep       217
Cabin           199
Destination     182
Age             179
VIP             203
RoomService     181
FoodCourt       183
ShoppingMall    208
Spa             183
VRDeck          188
Name            200
Transported       0
dtype: int64


In [18]:
# compruebo si hay duplicados
print("\nFilas Duplicadas:")
print(df.duplicated().sum())


Filas Duplicadas:
0


---
### Ingeniería de variables:

La columna 'Cabin' tiene tres valores, que parece que podrían descomponerse en cubierta, número y lado. La convierto en tres nuevas columnas a través de la división del string. Además, elimino la columna original y convierto el número de cabina en float.

In [19]:
# separo la columna 'Cabin' en tres nuevas variables: cubierta, número y lado
df[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']] = df['Cabin'].str.split('/', expand=True)

# elimino la columna original
df = df.drop(columns=['Cabin'])

# convierto la columna de número de cabina a tipo numérico
df['Cabin_Num'] = pd.to_numeric(df['Cabin_Num'])

# reviso cómo han quedado mis nuevas columnas
df[['Cabin_Deck', 'Cabin_Num', 'Cabin_Side']].head()

,Cabin_Deck,Cabin_Num,Cabin_Side
0,B,0.0,P
1,F,0.0,S
2,A,0.0,S
3,A,0.0,S
4,F,1.0,S


---
### Trabajo con valores nulos

Identifico qué columnas son numéricas, en una lista, y relleno nulos con la mediana. En el caso de las variables categóricas, lo hago con la moda, es decir, con el valor más repetido. La columna nombre de pasajero la quito, ya que no parece que aporte en la predicción. Y, por último, compruebo si quedan valores nulos.

In [20]:
# identifico las columnas numéricas para rellenarlas con su mediana
columnas_numericas = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Cabin_Num']

for col in columnas_numericas:
    mediana = df[col].median() # calculo la mediana de la columna
    df[col] = df[col].fillna(mediana) # relleno los nulos

# identifico las columnas categóricas para rellenarlas con su moda
columnas_categoricas = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Cabin_Deck', 'Cabin_Side']

for col in columnas_categoricas:
    moda = df[col].mode()[0] # calculo la moda y elijo el primer resultado
    df[col] = df[col].fillna(moda) # relleno los NaN

# elimino la columna del nombre del pasajero 
df = df.drop(columns=['Name'])

# compruebo que ya no me queda ningún valor nulo
df.isnull().sum()

PassengerId     0
HomePlanet      0
CryoSleep       0
Destination     0
Age             0
VIP             0
RoomService     0
FoodCourt       0
ShoppingMall    0
Spa             0
VRDeck          0
Transported     0
Cabin_Deck      0
Cabin_Num       0
Cabin_Side      0
dtype: int64

---
### Transformación de variables categóricas en numéricas:

Convierto las variables de True/False en 1/0. Uso el one hot encoding para las variables con varias clases. Compruebo cómo han quedado los registros después del procesado.

In [21]:
# convierto las variables de verdadero/falso a unos y ceros numéricos
columnas_booleanas = ['CryoSleep', 'VIP']
for col in columnas_booleanas:
    df[col] = df[col].astype(int) # cambio el tipo a entero

# selecciono las columnas con texto que tienen varias clases
columnas_categoricas = ['HomePlanet', 'Destination', 'Cabin_Deck', 'Cabin_Side']

# one hot encoding de las columnas categóricas
df = pd.get_dummies(df, columns=columnas_categoricas, drop_first=True)

# la columna objetivo 'Transported' también se convierte a 1 y 0, pero como no es variable predictora lo hago aparte
df['Transported'] = df['Transported'].astype(int)

# veo las primeras filas para comprobar que todo el dataset es numérico
df.head()

,PassengerId,CryoSleep,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Transported,...,Destination_PSO J318.5-22,Destination_TRAPPIST-1e,Cabin_Deck_B,Cabin_Deck_C,Cabin_Deck_D,Cabin_Deck_E,Cabin_Deck_F,Cabin_Deck_G,Cabin_Deck_T,Cabin_Side_S
0,0001_01,0,39.0,0,0.0,0.0,0.0,0.0,0.0,0,...,False,True,True,False,False,False,False,False,False,False
1,0002_01,0,24.0,0,109.0,9.0,25.0,549.0,44.0,1,...,False,True,False,False,False,False,True,False,False,True
2,0003_01,0,58.0,1,43.0,3576.0,0.0,6715.0,49.0,0,...,False,True,False,False,False,False,False,False,False,True
3,0003_02,0,33.0,0,0.0,1283.0,371.0,3329.0,193.0,0,...,False,True,False,False,False,False,False,False,False,True
4,0004_01,0,16.0,0,303.0,70.0,151.0,565.0,2.0,1,...,False,True,False,False,False,False,True,False,False,True


---
### Divido los datos en entrenamiento y test

In [22]:
# dataframe de variables predictoras (todas menos 'PassengerId' y la target)
x = df.drop(columns=['PassengerId', 'Transported'])

# target
y = df['Transported']

# divido los bloques en un 80% para entrenamiento y un 20% para la evaluación
# fijo una semilla
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=11)

# imprimo el tamaño de los bloques para comprobar que se han dividido bien
print(f"filas para que mi red entrene: {x_train.shape[0]}")
print(f"filas para examinar a mi red: {x_test.shape[0]}")

filas para que mi red entrene: 6954
filas para examinar a mi red: 1739


---
### Diseño y construcción de la red neuronal

In [29]:
# guardo el número exacto de variables predictoras que tiene mi bloque x
dimension_entrada = x_train.shape[1]

# configuro mi red guardando todas las capas ordenadas dentro de una lista
modelo_red = keras.Sequential([
    # defino el tamaño de la entrada usando el número de columnas de mi x_train
    keras.Input(shape=(x_train.shape[1],)),
    
    # añado mi primera capa oculta con 32 neuronas y la función de activación relu
    layers.Dense(32, activation='relu'),
    
    # añado una segunda capa oculta con 16 neuronas para buscar relaciones complejas
    layers.Dense(16, activation='relu'),
    
    # pongo la capa de salida con una sola neurona y sigmoide para la probabilidad
    layers.Dense(1, activation='sigmoid')
])

# pido ver la estructura y el resumen de conexiones de mi red recién creada
modelo_red.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,249 (4.88 KB)

 Trainable params: 1,249 (4.88 KB)

 Non-trainable params: 0 (0.00 B)

---
### Compilación de la red neuronal

In [30]:
# configuro los parámetros
modelo_red.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

---
### Entrenamiento de la red neuronal

In [31]:
historial = modelo_red.fit(
    x_train, y_train,
    epochs=20,
    batch_size=32,
    validation_data=(x_test, y_test)
)

Epoch 1/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6654 - loss: 16.2018 - val_accuracy: 0.7510 - val_loss: 3.1293
Epoch 2/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7307 - loss: 1.8814 - val_accuracy: 0.6596 - val_loss: 1.6623
Epoch 3/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7360 - loss: 1.2829 - val_accuracy: 0.6699 - val_loss: 1.3429
Epoch 4/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7498 - loss: 1.0075 - val_accuracy: 0.7384 - val_loss: 1.0512
Epoch 5/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7436 - loss: 0.9478 - val_accuracy: 0.7516 - val_loss: 0.9556
Epoch 6/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7473 - loss: 0.8865 - val_accuracy: 0.7832 - val_loss: 0.8621
Epoch 7/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7465 - loss: 0.8270 - val_accuracy: 0.7602 - val_loss: 0.8821
Epoch 8/20
218/218 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.7416 - loss: 0.8345 - val_accuracy: 0